# MD-CLARA on pairfam: comparing IDCD, CAT, and DAT

This notebook runs **all three multidomain strategies** on the same **1,027 pairfam respondents**, using:

- **Activity** (`pairfam_activity_by_month`) — 8 employment/activity states, monthly ages 18–40
- **Family** (`pairfam_family_by_month`) — 9 partnership/parenthood states, same calendar months

See [md_clara_pairfam_tutorial.ipynb](md_clara_pairfam_tutorial.ipynb) for data background and state tables.  
Patent-city example (3 domains): [md_clara_patent_cities_strategies.ipynb](md_clara_patent_cities_strategies.ipynb).

All runs share `R`, `sample_size`, `kvals`, and `random_state` so differences come from the **dissimilarity strategy**, not CLARA sampling.

## 1. Load data and build domains

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import adjusted_rand_score

from sequenzo import SequenceData, load_dataset
from sequenzo.multidomain.clara import (
    md_clara,
    plot_md_clara_quality,
    plot_md_clara_stability,
)

%matplotlib inline

df_activity = load_dataset("pairfam_activity_by_month")
df_family = load_dataset("pairfam_family_by_month")
assert (df_activity["id"].values == df_family["id"].values).all()

COVARIATE_COLS = [
    "id", "weight40", "sex", "doby_gen", "dob", "ethni", "migstatus", "yeduc",
    "sat1i4", "sat5", "sat6", "highschool", "church", "biosib", "stepsib",
    "east", "famstructure18",
]
TIME_COLS = [c for c in df_activity.columns if c not in COVARIATE_COLS]
person_meta = df_activity[COVARIATE_COLS].copy()
weights = df_activity["weight40"].values

ACTIVITY_STATES = list(range(1, 9))
ACTIVITY_LABELS = [
    "Education", "Military/Civil service", "Part-time", "Full-time",
    "Self-employed", "Parental leave", "Marginal employment", "Unemployed",
]
FAMILY_STATES = list(range(1, 10))
FAMILY_LABELS = [
    "Single, no child", "LAT, no child", "Cohabiting, no child", "Married, no child",
    "Single, with child(ren)", "LAT, with child(ren)", "Cohabiting, with child(ren)",
    "Married, 1 child", "Married, 2+ children",
]

domains = [
    SequenceData(
        df_activity, time=TIME_COLS, id_col="id",
        states=ACTIVITY_STATES, labels=ACTIVITY_LABELS, weights=weights,
    ),
    SequenceData(
        df_family, time=TIME_COLS, id_col="id",
        states=FAMILY_STATES, labels=FAMILY_LABELS, weights=weights,
    ),
]
print(f"N = {len(person_meta)}, months = {len(TIME_COLS)}")

## 2. Shared settings

In [ ]:
N_DOM = len(domains)

CLARA_KWARGS = dict(
    R=20,
    sample_size=200,
    kvals=[2, 3, 4, 5, 6],
    criteria=("distance",),
    stability=True,
    random_state=42,
    n_jobs=-1,
    verbose=True,
)

STRATEGY_CONFIG = {
    "idcd": {
        "label": "IDCD (joint activity–family states)",
        "distance_params": {
            "method": "OM", "sm": "CONSTANT", "indel": 1, "norm": "none",
        },
    },
    "cat": {
        "label": "CAT (additive multidomain costs)",
        "distance_params": {
            "method": "OM",
            "sm": ["CONSTANT"] * N_DOM,
            "indel": 1,
            "norm": "none",
        },
    },
    "dat": {
        "label": "DAT (sum of domain distances)",
        "distance_params": {
            "method_params": [
                {"method": "OM", "sm": "CONSTANT", "indel": 1},
            ] * N_DOM,
            "domain_weights": [1.0] * N_DOM,
            "link": "sum",
        },
    },
}

## 3. Run all three strategies

Expect longer runtime than the patent-city tutorials because each sequence has **264** time points.

In [ ]:
results = {}

for strategy, cfg in STRATEGY_CONFIG.items():
    print("\n" + "=" * 60)
    print(cfg["label"])
    print("=" * 60)
    results[strategy] = md_clara(
        domains,
        strategy=strategy,
        distance_params=cfg["distance_params"],
        **CLARA_KWARGS,
    )

## 4. Compare quality statistics

`avg_dist` scales differ across strategies; compare curves **within** a strategy and use **ARI** below to compare **partitions** across strategies.

In [ ]:
quality = pd.concat(
    [res.stats.assign(strategy=s) for s, res in results.items()],
    ignore_index=True,
)
quality[["strategy", "k", "avg_dist", "db", "xb", "ari08", "jc08"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for strategy, cfg in STRATEGY_CONFIG.items():
    sub = quality[quality["strategy"] == strategy]
    ax.plot(sub["k"], sub["avg_dist"], marker="o", label=cfg["label"])
ax.set_xlabel("k")
ax.set_ylabel("Average distance to medoids")
ax.set_title("Within-strategy fit (pairfam activity + family)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Quality and stability plots

In [ ]:
for strategy, res in results.items():
    title = STRATEGY_CONFIG[strategy]["label"]
    plot_md_clara_quality(res, title=f"{title} — quality")
    plt.show()
    plot_md_clara_stability(res, title=f"{title} — stability")
    plt.show()

## 6. Agreement between strategies at chosen k

In [ ]:
K_CHOSEN = 4
strategies = list(STRATEGY_CONFIG.keys())
labels = {s: results[s].best_clustering(K_CHOSEN) for s in strategies}

ari_matrix = pd.DataFrame(index=strategies, columns=strategies, dtype=float)
for a in strategies:
    for b in strategies:
        ari_matrix.loc[a, b] = adjusted_rand_score(labels[a], labels[b])

print(f"Adjusted Rand index at k = {K_CHOSEN}")
ari_matrix.round(3)

In [ ]:
compare = person_meta.copy()
for s in strategies:
    compare[f"cluster_{s}_k{K_CHOSEN}"] = labels[s]

pd.crosstab(
    compare[f"cluster_idcd_k{K_CHOSEN}"],
    compare[f"cluster_cat_k{K_CHOSEN}"],
    rownames=["IDCD"], colnames=["CAT"],
)

## 7. Export

In [ ]:
export = compare.copy()
for strategy, res in results.items():
    export = export.join(res.clustering.add_prefix(f"{strategy}_"))

out_path = Path("md_clara_pairfam_all_strategies.csv")
export.to_csv(out_path, index=False)
print(f"Saved: {out_path.resolve()}")

## 8. Choosing a strategy for life-course research

- **IDCD** — Emphasizes **joint** activity–family configurations actually observed (e.g. parental leave while married). Strong default when you expect work and family domains to be intertwined.
- **CAT** — Concatenation-style multidomain OM; useful when you want explicit additive substitution structure across domains.
- **DAT** — Sums domain-level distances; treats activity and family somewhat more separately. Can diverge from IDCD when domains are strongly associated.

For pairfam, activity and family are classic **interdependent** life domains; compare partitions (ARI) and substantive typologies before reporting a single strategy.

Docs: [pairfam activity](https://github.com/Liang-Team/SequenzoWebsite/blob/main/docs/en/datasets/pairfam-activity.md) · [pairfam family](https://github.com/Liang-Team/SequenzoWebsite/blob/main/docs/en/datasets/pairfam-family.md)